<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_23_practicum_recursion/note_lesson_23_recursion_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 23 — Практикум П4: рекурсія, «розділяй і володарюй», перебір з поверненням

Три задачі сервісу «Смачно + Таксі», у яких відповідь для цілого збирається з відповідей для частин:

1. **Вкладене меню** невідомої глибини: скільки страв, де «Узвар», скільки коштує все меню.
2. **Сортування доставок**: вставка з П1 — `O(n²)`, злиття — `O(n log n)`.
3. **Ваучер на будь-яку кількість поїздок**: Two Sum з П3 шукав пару, а тепер потрібні набори будь-якого розміру.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 1 | рекурсія: базовий і рекурсивний випадок, стек викликів, вкладене меню, шлях, межа глибини | 1.1–1.9 | **Задача 1** |
| 2 | розділяй і володарюй: злиття, сортування злиттям, дослід подвоєння, стабільність | 2.1–2.6 | **Задача 2** |
| 3 | перебір з поверненням: дерево рішень, усі підмножини, відсікання | 3.1–3.6 | **Задача 3** |
| 4 | коли рекурсія повторює роботу: мемоізація, `lru_cache` | 4.1–4.5 | |
| 5 | рекурсія чи цикл: явний стек, дуже глибоке меню | 5.1–5.3 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, константи) лише створює функцію чи дані — **запуск завжди в наступній клітинці**.
- Багато прикладів друкують **трасу**: кожен рекурсивний виклик — на рядок нижче й з більшим відступом. Відступ = глибина стеку викликів.
- Перед **🔮 Прогнозом** спершу скажи, що буде, потім розгортай відповідь.
- Задач три. Розв'язок пишеш у клітинці з визначенням, перевірка — у наступній клітинці; вона друкує «✅ Задача N розв'язана».
- Теорія, дерева рішень і «рекурсія чи цикл» — у книзі: [Урок 23. Практикум П4](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_23/).

## 🔁 Пригадай (без підглядання)

1. Скільки кроків бінарного пошуку на 1 000 000 записів?
2. Що знаходив Two Sum у П3?
3. Що робить `@lru_cache`?

<details>
<summary>Відповіді</summary>

1. Близько 20.
2. Пару чисел із заданою сумою за один прохід.
3. Запам'ятовує результати функції.

</details>

---
## 1. Рекурсія: функція, що викликає сама себе

Кожна рекурсивна функція має дві частини:

- **базовий випадок** — задача настільки мала, що відповідь відома одразу. Тут рекурсія **зупиняється**;
- **рекурсивний випадок** — задачу зводимо до **меншої** задачі того самого виду й використовуємо її відповідь.

### Приклад 1.1. Стек викликів: вхід і вихід

Найменша рекурсія — зворотний відлік. Друкуємо рядок **до** рекурсивного виклику і **після** нього, з відступом за глибиною.

In [ ]:
from pathlib import Path

print(Path.cwd())

## Практичний приклад: рекурсивний пошук і читання файлу

### Мета

Навчитися використовувати **рекурсію для пошуку файлів у вкладених папках**, зрозуміти принцип роботи **стеку викликів** та навчитися відкривати й читати Jupyter Notebook (`.ipynb`).

### 1. Яку проблему розв'язуємо?

Уявімо, що маємо проєкт із такою структурою:

```text
PY-Course/
├── module_1/
│   ├── lesson_1/
│   └── lesson_2/
├── module_2/
│   └── lessons/
│       ├── lesson_21/
│       ├── lesson_22/
│       └── lesson_23_practicum_recursion/
│           └── note_lesson_23_recursion.ipynb
└── module_3/
```

Потрібно знайти файл `note_lesson_23_recursion.ipynb`, не знаючи заздалегідь, у якій саме папці та на якій глибині він розташований.

Звичайний цикл `for` може перебрати елементи однієї папки. Але що робити, якщо всередині неї є інші папки, а в них — ще папки?

**Рішення — рекурсія:** функція заходить у кожну знайдену папку, викликаючи саму себе.

### 2. Як працює функція `find_file()`?

Функція приймає два аргументи:

- `folder` — папка, у якій зараз виконується пошук.
- `filename` — назва файлу, який потрібно знайти.

**Крок 1. Перебір елементів папки**

```python
for item in sorted(folder.iterdir()):
```

- `folder.iterdir()` повертає файли та папки всередині поточної директорії.
- `sorted()` упорядковує елементи, щоб порядок обходу був передбачуваним.
- `item` — поточний файл або папка.

**Крок 2. Перевірка файлу — базовий випадок успішного пошуку**

```python
if item.is_file() and item.name == filename:
```

Перевіряємо, чи є поточний елемент файлом і чи збігається його назва з потрібною.

Якщо файл знайдено:

1. Виводимо його шлях.
2. Відкриваємо файл за допомогою `item.open()`.
3. Використовуємо `json.load(f)`, щоб прочитати структуру Jupyter Notebook.
4. Перебираємо комірки `notebook["cells"]`.
5. Виводимо тип та текст кожної комірки.
6. Повертаємо `True`, щоб завершити пошук.

**Крок 3. Рекурсивний виклик**

```python
elif item.is_dir():
    if find_file(item, filename):
        return True
```

Якщо `item` — папка, функція викликає саму себе для цієї папки.

Це і є **рекурсивний випадок**.

Поточний виклик тимчасово призупиняється, а Python починає виконувати новий виклик `find_file()`.

Якщо вкладений виклик знайшов файл і повернув `True`, поточний виклик також повертає `True`.

**Крок 4. Повернення, якщо файл не знайдено**

```python
print(f"⬅️ Повертаємось із: {folder.name}")
return False
```

Якщо всі елементи поточної папки перевірено, але потрібний файл не знайдено, функція повертає `False`.

Це означає, що Python повертається до попереднього виклику й продовжує пошук в інших папках.

### 3. Як працює стек викликів?

Кожен рекурсивний виклик додає новий запис до **стеку викликів (Call Stack)**.

Припустимо, пошук проходить таким шляхом:

```text
find_file(PY-Course)
    │
    └── find_file(module_2)
            │
            └── find_file(lessons)
                    │
                    └── find_file(lesson_23_practicum_recursion)
                            │
                            └── ✅ ФАЙЛ ЗНАЙДЕНО!
```

У момент знаходження файлу в стеку перебувають чотири виклики функції.

Після `return True` виклики завершуються у зворотному порядку:

```text
lesson_23_practicum_recursion → True
lessons                      → True
module_2                     → True
PY-Course                    → True
```

Це принцип **LIFO (Last In, First Out)** — останній виклик завершується першим.

Зверніть увагу: повідомлення `⬅️ Повертаємось із:` у нашому коді друкується лише тоді, коли в поточній папці файл не знайдено. Під час успішного пошуку `return True` завершує функцію раніше, тому ці повідомлення можуть не з'явитися, хоча повернення через стек відбувається.

### 4. Чим відрізняються `print()` і `return`?

| `print()` | `return` |
|---|---|
| Виводить інформацію на екран | Повертає результат виклику |
| Не завершує функцію автоматично | Завершує поточний виклик функції |
| Використовується для повідомлень | Використовується для передачі результату |

У нашій функції `print()` показує процес пошуку та вміст файлу, а `return True` повідомляє попередньому виклику, що пошук завершено успішно.

### 5. Чому тут використовується `json`?

Jupyter Notebook (`.ipynb`) — це файл у форматі JSON. Його структура містить список комірок:

```json
{
  "cells": [
    {
      "cell_type": "markdown",
      "source": ["# Рекурсія"]
    },
    {
      "cell_type": "code",
      "source": ["print('Hello, Python!')"]
    }
  ]
}
```

Тому:

```python
notebook = json.load(f)
```

перетворює JSON на об'єкти Python, а:

```python
for number, cell in enumerate(notebook["cells"], start=1):
    print("".join(cell["source"]))
```

дозволяє прочитати текст усіх комірок.

### 6. Завдання для самостійної роботи

1. Додайте параметр `depth=0`, щоб показувати рівень вкладеності папок.
2. Змініть функцію так, щоб вона знаходила **всі `.py` файли**, а не один конкретний.
3. Додайте лічильник кількості перевірених папок.
4. Спробуйте знайти файл, якого не існує. Простежте, коли функція повертає `False`.
5. Змініть функцію так, щоб вона повертала шлях знайденого файлу (`Path`), а не `True`.

### Висновок

**Рекурсія дозволяє обробляти вкладені структури без необхідності заздалегідь знати їхню глибину.**

У цьому прикладі кожна папка — це нова задача того самого типу: знайти файл серед її елементів і вкладених папок.

Python використовує стек викликів, щоб зберігати незавершені виклики й повертатися до них після виконання вкладених функцій.

In [ ]:
from pathlib import Path
import json

def find_file(folder, filename):
    print(f"📂 Перевіряємо: {folder}")

    for item in sorted(folder.iterdir()):

        if item.is_file() and item.name == filename:
            print(f"\n✅ ЗНАЙДЕНО: {item}\n")

            # Відкриваємо Jupyter Notebook
            with item.open("r", encoding="utf-8") as f:
                notebook = json.load(f)

            # Друкуємо всі комірки
            for number, cell in enumerate(notebook["cells"], start=1):
                print(f"\n{'=' * 50}")
                print(f"КОМІРКА {number} | {cell['cell_type']}")
                print("=" * 50)
                print("".join(cell["source"]))

            return True

        elif item.is_dir():
            if find_file(item, filename):
                return True

    print(f"⬅️ Повертаємось із: {folder.name}")
    return False


In [ ]:
# Знайти корінь проєкту серед поточної папки та її батьків
current = Path.cwd()
project = next(
    (p for p in (current, *current.parents)
     if p.name == "PY-Course-Victor-Nikoriak-22-09-2026"),
    None
)


In [ ]:
if project is None:
    print("❌ Корінь проєкту не знайдено:", current)
else:
    result = find_file(project, "note_lesson_23_recursion.ipynb")

    if result is None:
        print("❌ Файл не знайдено")
    else:
        print("✅ Файл знайдено:", result)

In [ ]:
from pathlib import Path
import runpy

root = Path(
    r"C:\Users\victo\PycharmProjects\PY-Course-Victor-Nikoriak-22-09-2026"
)

In [ ]:
print(root)

In [ ]:
def find_file(folder, filename, depth=1):
    indent = "    " * depth
    print(f"{indent}📂 {folder.name}")
    for item in sorted(folder.iterdir()):
        if item.is_file() and item.name == filename:
            print(f"{indent}✅ ЗНАЙДЕНО: {item.resolve()}")
            return item
        elif item.is_dir():
            # Пропускаємо службові папки
            if item.name in {".venv", ".git", ".idea", "__pycache__"}:
                continue
            result = find_file(item, filename, depth + 1)
            if result is not None:
                return result
    return None

In [ ]:
def open_file(root, filename):
    print("🔎 Починаємо пошук...")
    if not root.is_dir():
        print("❌ Коренева папка не існує:", root)
        return
    file = find_file(root, filename)

    if file is None:
        print("❌ Файл не знайдено!")
        return
    print(f"\n🚀 ЗАПУСКАЄМО: {file.resolve()}\n")
    runpy.run_path(
        str(file.resolve()),
        run_name="__main__"
    )
    print("\n✅ Виконання модуля завершено")

In [ ]:
open_file(root, "recursion.py")

In [ ]:
project = Path(
    r"C:\Users\victo\PycharmProjects\PY-Course-Victor-Nikoriak-22-09-2026"
)

print("Проєкт існує:", project.is_dir())

In [ ]:
from PIL import Image
from IPython.display import display

image_path = find_file(project, "recurs.jpg")

if image_path is not None:
    picture = Image.open(image_path)
    display(picture)

In [ ]:
def countdown_traced(n, depth=0):
    pad = "    " * depth
    print(f"{pad}→ вхід у countdown({n})")
    if n == 0:
        print(f"{pad}  базовий випадок: далі не йдемо")
        return
    countdown_traced(n - 1, depth + 1)
    print(f"{pad}← вихід з countdown({n})")

In [ ]:
countdown_traced(3)

Входи йдуть 3 → 2 → 1 → 0, а виходи — у **зворотному** порядку: 1 → 2 → 3. Кожен виклик — окремий **кадр** (frame) з власним `n`. Кадри складаються в **стек**: новий кладеться зверху, а знімається першим той, що поклали останнім. `countdown(3)` не може завершитись, поки не завершиться `countdown(2)`.

### Приклад 1.2. Відповідь піднімається знизу вгору

Сума поїздок зміни: сума списку = перша поїздка + сума **решти**. Базовий випадок — порожній список, сума 0.

In [ ]:
def trip_total(fares, depth=0):
    pad = "    " * depth
    if not fares:
        print(f"{pad}trip_total([]) = 0  ← базовий випадок")
        return 0
    rest = trip_total(fares[1:], depth + 1)
    result = fares[0] + rest
    print(f"{pad}trip_total({fares}) = {fares[0]} + {rest} = {result}")
    return result

In [ ]:
print("Разом:", trip_total([230, 150, 270]))

Траса читається знизу вгору: спершу базовий випадок дав 0, потім кожен кадр додав свою поїздку до відповіді «нижнього» кадру: 270, 420, 650. Для списку це надто складно — є `sum()`. Але для **вкладених** даних простішого способу немає.

### Приклад 1.3. Скільки страв у вкладеному меню

Меню кафе — вкладені словники. Лист — страва з ціною, вузол — категорія. Цикл `for` не допоможе: невідомо, скільки рівнів вкладень. Зате відповідь легко сказати **через саму себе**: страв у категорії — сума страв у кожній її частині. А частина — або страва (1), або знову категорія.

In [ ]:
MENU = {
    "Кухня": {
        "Перші страви": {"Борщ": 95, "Юшка": 85},
        "Основні": {"Вареники": 80, "Деруни": 75},
    },
    "Напої": {
        "Гарячі": {"Чай": 30, "Кава": {"Еспресо": 40, "Лате": 55}},
        "Холодні": {"Узвар": 35},
    },
    "Хліб": 10,
}


def count_dishes(node):
    if not isinstance(node, dict):
        return 1
    return sum(count_dishes(child) for child in node.values())

**🔮 Прогноз:** скільки страв у меню? (рахуй лише листки з цінами)

In [ ]:
print(count_dishes(MENU))
print(count_dishes(MENU["Напої"]))
print(count_dishes(95))

<details>
<summary>Відповідь</summary>

9. Базовий випадок — страва (1), рекурсивний — категорія (сума по частинах).

</details>

### Приклад 1.4. Траса обходу меню

Та сама функція, але з відступами: так видно, у якому порядку Python заходить у категорії й звідки береться кожне число.

In [ ]:
def count_dishes_traced(node, name="Меню", depth=0):
    pad = "    " * depth
    if not isinstance(node, dict):
        print(f"{pad}{name} → 1")
        return 1
    print(f"{pad}{name}")
    total = sum(count_dishes_traced(child, child_name, depth + 1) for child_name, child in node.items())
    print(f"{pad}{name} = {total}")
    return total

In [ ]:
print("Відповідь:", count_dishes_traced(MENU["Напої"], "Напої"))

Рядки «→ 1» — базові випадки: там стек перестає рости. Відповіді піднімаються знизу вгору: «Кава» = 2, «Гарячі» = 1 + 2 = 3, «Напої» = 3 + 1 = 4. Найглибший відступ — у «Еспресо» й «Лате»: у всьому меню це п'ять кадрів один над одним (Меню → Напої → Гарячі → Кава → Лате).

```text
Напої (4)
├── Гарячі (3)
│   ├── Чай → 1
│   └── Кава (2)
│       ├── Еспресо → 1
│       └── Лате → 1
└── Холодні (1)
    └── Узвар → 1
```

### Приклад 1.5. Рекурсія, що повертає шлях

Де в меню «Узвар»? Функція повертає шлях категорій або `None`, якщо страви немає. Шлях передається **параметром**: кожен рівень додає своє ім'я до кортежу й передає новий кортеж глибше.

In [ ]:
def find_path(node, dish, path=()):
    if not isinstance(node, dict):
        return None
    for name, child in node.items():
        if name == dish:
            return path + (name,)
        found = find_path(child, dish, path + (name,))
        if found:
            return found
    return None

In [ ]:
print(find_path(MENU, "Узвар"))
print(find_path(MENU, "Лате"))
print(find_path(MENU, "Піца"))

In [ ]:
from bs4 import BeautifulSoup

html = """
<div class="container">
    <p>Прямий нащадок</p>
    <div class="inner">
        <p>Вкладений нащадок</p>
    </div>
</div>
"""

soup = BeautifulSoup(html, "html.parser")
container = soup.find("div", class_="container")

# Знайде обидва теги <p> (recursive=True за замовчуванням)
print(len(container.find_all("p")))  # 2

# Знайде лише перший тег <p> без занурення у <div class="inner">
print(len(container.find_all("p", recursive=False)))  # 1


Для «Піци» всі гілки повернули `None`, і верхній виклик теж повернув `None`.

### Приклад 1.6. Чому шлях — кортеж, а не список з `append`

Спокуса: тримати шлях у списку й дописувати в нього `append`. Ось що буде.

In [ ]:
def find_path_buggy(node, dish, path=[]):        # змінюване значення за замовчуванням!
    if not isinstance(node, dict):
        return None
    for name, child in node.items():
        path.append(name)                         # один список на ВСІ гілки й виклики
        if name == dish:
            return path
        found = find_path_buggy(child, dish, path)
        if found:
            return found
    return None

In [ ]:
print(find_path_buggy(MENU, "Узвар"))
second = find_path_buggy(MENU, "Хліб")
print(len(second), "імен у «шляху» до Хліба")

Дві помилки одразу:

- «шлях» до Узвару містить усе, що функція **відвідала** дорогою: Борщ, Юшку, Вареники… Гілки не прибирають за собою свої імена, бо пишуть в один список;
- другий виклик продовжив **той самий** список: значення за замовчуванням створюється один раз, при визначенні функції (урок 8).

`path + (name,)` щоразу створює **новий** кортеж, тож у кожної гілки свій шлях. (У розділі 3 ми свідомо використаємо один список — але з `pop()`, який прибирає за собою.)

### Приклад 1.7. Межа глибини

Кожен кадр займає пам'ять, тож Python обмежує глибину стеку.

**🔮 Прогноз:** чи впаде `countdown(100_000)`?

In [ ]:
import sys


def countdown(n):
    if n == 0:
        return 0
    return countdown(n - 1)

In [ ]:
print(sys.getrecursionlimit())
print(countdown(500))
try:
    countdown(100_000)
except RecursionError as error:
    print(type(error).__name__, "—", error)

<details>
<summary>Відповідь</summary>

Так, `RecursionError`: глибина стеку обмежена. Перше число — ліміт: у звичайному Python це 1000 кадрів, а Jupyter і Colab часто піднімають його (наприклад, до 3000). Точне число залежить від середовища; важливо, що воно **скінченне**, а 100 000 його точно перевищує. Для лінійних даних — цикл.

</details>

### Приклад 1.8. Базовий випадок, якого не досягнемо

Мало **мати** базовий випадок — до нього треба **дійти**. Дві типові помилки: задача не зменшується, або ми «перестрибуємо» через базу.

In [ ]:
def countdown_stuck(n):
    if n == 0:
        return 0
    return countdown_stuck(n)          # забули зменшити n

In [ ]:
for label, call in (("countdown_stuck(3)", lambda: countdown_stuck(3)),
                    ("countdown(-1)", lambda: countdown(-1))):    # -1, -2, -3 … ніколи не 0
    try:
        call()
    except RecursionError:
        print(f"{label}: RecursionError")

Обидва виклики впали з тим самим `RecursionError`: у першому `n` не змінюється, у другому `n` віддаляється від нуля. Перевір себе для кожної рекурсії: **кожен** виклик має наближати до базового випадку.

### Приклад 1.9. Звіт за меню: один шаблон — три відповіді

Скільки коштує все меню, яка страва найдешевша, скільки рівнів вкладеності? Три маленькі рекурсії одного шаблону: базовий випадок — страва, рекурсивний — **згортка** відповідей частин (`sum`, `min`, `max`). Це reducer з уроку 8, застосований до дерева.

In [ ]:
def total_price(node):
    if not isinstance(node, dict):
        return node
    return sum(total_price(child) for child in node.values())


def cheapest(node):
    if not isinstance(node, dict):
        return node
    return min(cheapest(child) for child in node.values())


def depth(node):
    if not isinstance(node, dict):
        return 0
    return 1 + max(depth(child) for child in node.values())

In [ ]:
print("Страв:", count_dishes(MENU), "| усе меню:", total_price(MENU), "грн")
print("Найдешевша страва кухні:", cheapest(MENU["Кухня"]), "грн | у всьому меню:", cheapest(MENU), "грн")
print("Рівнів вкладеності:", depth(MENU), "| у страви:", depth(95))
assert total_price(MENU) == 505 and cheapest(MENU["Кухня"]) == 75 and depth(MENU) == 4
assert cheapest(MENU) == 10 and depth(95) == 0

- `total_price` і `cheapest` відрізняються лише згорткою: `sum` чи `min`;
- `depth`: категорія на один рівень глибша за свою найглибшу частину. Меню → Напої → Гарячі → Кава → страва — 4 рівні категорій;
- жодна функція не знає, скільки рівнів у меню. Додай «Десерти → Торти → Шоколадні» — код не зміниться.

### Задача 1. Генератор страв зі шляхами

Напиши **рекурсивний генератор** `dishes(node, path=())` (урок 11), який видає пари `(шлях, ціна)` для кожної страви меню в порядку меню:

```text
next(dishes(MENU))              →  (('Кухня', 'Перші страви', 'Борщ'), 95)
len(list(dishes(MENU)))         →  9
```

- базовий випадок: `node` — не словник → `yield path, node` і `return`;
- рекурсивний: для кожної пари `name, child` — `yield from dishes(child, path + (name,))` (шлях — кортеж, як у прикладі 1.5).

Генератор знадобиться ще в задачі 3. Перевірка — у наступній клітинці.

In [ ]:
def dishes(node, path=()):
    # YOUR CODE HERE
    pass


In [ ]:
print(next(dishes(MENU)))
assert next(dishes(MENU)) == (("Кухня", "Перші страви", "Борщ"), 95)
assert len(list(dishes(MENU))) == 9
assert (("Напої", "Гарячі", "Кава", "Лате"), 55) in list(dishes(MENU))
assert sum(1 for _ in dishes(MENU)) == count_dishes(MENU)
assert list(dishes(10)) == [((), 10)], "одна страва без категорій — порожній шлях"
print("✅ Задача 1 розв'язана")

---
## 2. Розділяй і володарюй: сортування злиттям

В П1 ми бачили: алгоритм, що порівнює кожен елемент з кожним, росте як `O(n²)`. Сортування вставкою — саме таке. **Розділяй і володарюй** (divide and conquer) пропонує інше:

1. **розділити** список навпіл;
2. **відсортувати** кожну половину — рекурсивно, тим самим алгоритмом;
3. **злити** дві відсортовані половини в одну.

Базовий випадок — список з 0 чи 1 елемента вже відсортований. Уся робота — у злитті.

### Приклад 2.1. Злиття двох відсортованих списків

Дивимося на перші елементи обох списків, менший переносимо в результат. Коли один список закінчився, решту другого дописуємо як є. `counter` — список з одного числа: лічильник порівнянь, спільний для всіх викликів (число всередині функції змінити не вийшло б — це локальна змінна, урок 19).

In [ ]:
def merge(left, right, counter):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        counter[0] += 1
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    return result + left[i:] + right[j:]

In [ ]:
counter = [0]
print(merge([150, 230, 270], [120, 180, 410], counter), "| порівнянь:", counter[0])

5 порівнянь на 6 елементів: щойно `left` закінчився, 410 дописано без порівнянь. Злиття проходить кожен елемент **один раз** — `O(n)`.

### Приклад 2.2. Сортування злиттям

In [ ]:
def merge_sort(items, counter):
    if len(items) <= 1:
        return list(items)
    middle = len(items) // 2
    left = merge_sort(items[:middle], counter)
    right = merge_sort(items[middle:], counter)
    return merge(left, right, counter)

In [ ]:
fares = [230, 150, 270, 180, 320, 120, 410, 150]
counter = [0]
print(merge_sort(fares, counter), counter[0])

8 доставок відсортовано за 16 порівнянь.

### Приклад 2.3. Траса: поділ униз, злиття вгору

In [ ]:
def merge_sort_traced(items, depth=0):
    pad = "    " * depth
    if len(items) <= 1:
        print(f"{pad}{items}  ← база")
        return list(items)
    middle = len(items) // 2
    print(f"{pad}{items}  ділимо")
    left = merge_sort_traced(items[:middle], depth + 1)
    right = merge_sort_traced(items[middle:], depth + 1)
    merged = merge(left, right, [0])
    print(f"{pad}{merged}  злили")
    return merged

In [ ]:
sorted_fares = merge_sort_traced(fares)

Відступи показують **рівні** поділу: 8 → 4 → 2 → 1, тобто `log₂ 8 = 3` рівні над базою. На кожному рівні злиття разом переглядає всі 8 елементів. Разом — `n · log n`.

```text
рівень 0:   [230 150 270 180 320 120 410 150]
рівень 1:   [230 150 270 180]   [320 120 410 150]
рівень 2:   [230 150] [270 180] [320 120] [410 150]
рівень 3:   по одному — база
злиття:     [150 230] [180 270] [120 320] [150 410] → [150 180 230 270] [120 150 320 410] → відсортовано
```

### Приклад 2.4. Дослід подвоєння: вставка проти злиття

Сортування вставкою з П1 — з тим самим лічильником порівнянь.

In [ ]:
def insertion_sort(items, counter):
    result = list(items)
    for k in range(1, len(result)):
        j = k
        while j > 0:
            counter[0] += 1
            if result[j - 1] <= result[j]:
                break
            result[j - 1], result[j] = result[j], result[j - 1]
            j -= 1
    return result

**🔮 Прогноз:** у скільки разів зростуть порівняння вставки й злиття, коли даних удвічі більше?

In [ ]:
for n in [1000, 2000, 4000]:
    data = list(range(n, 0, -1))         # зворотний порядок — найгірший випадок вставки
    slow, fast = [0], [0]
    insertion_sort(data, slow)
    merge_sort(data, fast)
    print(n, slow[0], fast[0])

<details>
<summary>Відповідь</summary>

Вставка — ×4 (`O(n²)`), злиття — трохи більше ніж ×2 (`O(n log n)`).

</details>

На 4000 доставок — 8 мільйонів порівнянь проти 24 тисяч. Вбудований `sorted()` — це Timsort, нащадок сортування злиттям: у житті сортуй ним, а `merge_sort` пишемо, щоб зрозуміти ідею.

### Приклад 2.5. Той самий шаблон для максимуму

«Розділяй і володарюй» — не лише сортування. Найдорожча поїздка = більша з найдорожчих у двох половинах.

In [ ]:
def max_fare(items, counter):
    if len(items) == 1:
        return items[0]
    middle = len(items) // 2
    left = max_fare(items[:middle], counter)
    right = max_fare(items[middle:], counter)
    counter[0] += 1
    return left if left >= right else right

In [ ]:
counter = [0]
print(max_fare(fares, counter), "| порівнянь:", counter[0], "| max():", max(fares))

410 за 7 порівнянь — рівно стільки ж, скільки робить звичайний цикл (n − 1). Для максимуму поділ нічого не виграв: кожну поїздку все одно треба побачити. Виграш сортування злиттям — у тому, що **злиття** відсортованих половин дешеве, а не в самому поділі.

### Приклад 2.6. Стабільність: рівні елементи не міняються місцями

Сортуємо доставки за ціною. Що з доставками з **однаковою** ціною? `merge` бере елемент з лівої половини, якщо ключі рівні (`<=`). Тому рівні елементи зберігають початковий порядок — сортування **стабільне**. Порівняємо з варіантом, де стоїть `<`.

In [ ]:
def merge_sort_by(items, key, stable=True):
    if len(items) <= 1:
        return list(items)
    middle = len(items) // 2
    left = merge_sort_by(items[:middle], key, stable)
    right = merge_sort_by(items[middle:], key, stable)
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        take_left = key(left[i]) <= key(right[j]) if stable else key(left[i]) < key(right[j])
        if take_left:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    return result + left[i:] + right[j:]


deliveries = [("Борщ і Ко", 150), ("Піца Поділ", 120), ("Суші Оболонь", 150), ("Вареники 24/7", 120)]

In [ ]:
by_price = lambda delivery: delivery[1]
print("<= :", merge_sort_by(deliveries, by_price))
print("<  :", merge_sort_by(deliveries, by_price, stable=False))
print("sorted:", sorted(deliveries, key=by_price))

З `<=` «Піца Поділ» лишилась перед «Вареники 24/7», а «Борщ і Ко» — перед «Суші Оболонь», як у вхідному списку. З `<` рівні елементи помінялися місцями. `sorted()` теж стабільний — тому можна сортувати в кілька проходів: спершу за назвою, потім за ціною, і в межах однієї ціни порядок за назвою збережеться.

### Задача 2. Бінарний пошук рекурсією

Бінарний пошук з П2 — теж «розділяй і володарюй», але йде лише в **одну** половину. Напиши рекурсивний `binary_search(items, target, low=0, high=None)`, який повертає індекс `target` у відсортованому списку або `-1`:

- розглядаємо проміжок `items[low:high]` (`high` не включно);
- база: порожній проміжок (`low >= high`) → `-1`;
- інакше дивимося на середину `middle = (low + high) // 2`: збіг — повертаємо `middle`; середина менша за `target` — шукаємо в правій половині `middle + 1 … high`, інакше — у лівій `low … middle`.

Перевірка — у наступній клітинці.

In [ ]:
def binary_search(items, target, low=0, high=None):
    if high is None:
        high = len(items)
    # YOUR CODE HERE
    pass


In [ ]:
starts = [425, 510, 612, 700, 845, 930, 1035, 1120, 1210, 1290, 1375]
assert binary_search(starts, 1120) == 7
assert binary_search(starts, 425) == 0 and binary_search(starts, 1375) == 10
assert binary_search(starts, 999) == -1 and binary_search([], 5) == -1
assert all(binary_search(starts, value) == index for index, value in enumerate(starts))
big = list(range(0, 2_000_000, 2))
assert binary_search(big, 1_234_566) == 617_283, "на мільйоні записів — близько 20 рівнів рекурсії"
print("✅ Задача 2 розв'язана")

---
## 3. Перебір з поверненням: ваучер на будь-яку кількість поїздок

У П3 ваучер покривав **дві** поїздки, і словник знаходив пару за один прохід. Тепер ваучер на 500 грн можна розділити між **скількома завгодно** поїздками зміни. Які набори дають рівно 500?

Кожна поїздка або входить у набір, або ні. **Перебір з поверненням** (backtracking) будує набір крок за кроком: «беру цю поїздку й дивлюся далі». Потім **повертається** на крок назад — прибирає поїздку — і пробує наступну:

- `chosen.append` — крок уперед: беремо поїздку;
- рекурсивний виклик досліджує все, що може піти після неї;
- `chosen.pop()` — **повернення**: прибираємо поїздку, щоб спробувати іншу на її місці.

Параметр `start` каже, що далі розглядаємо лише поїздки після поточної: так кожен набір з'являється один раз.

### Приклад 3.1. Усі набори на рівно 500 грн

In [ ]:
def voucher_sets(fares, target):
    found = []
    calls = [0]

    def explore(start, chosen, total):
        calls[0] += 1
        if total == target:
            found.append(list(chosen))
            return
        for i in range(start, len(fares)):
            chosen.append(fares[i])
            explore(i + 1, chosen, total + fares[i])
            chosen.pop()

    explore(0, [], 0)
    return found, calls[0]


trip_fares = [230, 150, 270, 180, 120, 410]

In [ ]:
print(voucher_sets(trip_fares, 500))

Два набори за 56 викликів: три поїздки 230 + 150 + 120 і дві 230 + 270. Друга — та сама пара, яку знайшов Two Sum у П3, а перша — те, чого пара знайти не могла. `found.append(list(chosen))` зберігає **копію**: сам `chosen` далі зміниться.

### Приклад 3.2. Дерево рішень у трасі

Та сама ідея на чотирьох поїздках, з відступом за глибиною: кожен рядок — один виклик `explore`, тобто один вузол дерева рішень.

In [ ]:
def voucher_traced(fares, target):
    def explore(start, chosen, total, depth):
        pad = "    " * depth
        mark = "  ✔ рівно!" if total == target else ("  ✘ забагато" if total > target else "")
        print(f"{pad}{chosen} = {total}{mark}")
        if total == target:
            return
        for i in range(start, len(fares)):
            chosen.append(fares[i])
            explore(i + 1, chosen, total + fares[i], depth + 1)
            chosen.pop()

    explore(0, [], 0, 0)

In [ ]:
voucher_traced([230, 150, 270, 120], 500)

- з кореня `[] = 0` розходяться чотири гілки — по одній на першу взяту поїздку;
- `[230, 270]` і `[230, 150, 120]` — дві відповіді (✔): на них гілка зупиняється;
- рядки з ✘ — сума вже більша за 500, але функція не знає, що далі буде лише гірше, і перебирає нащадків до кінця: під `[230, 150, 270] = 650` вона ще дослідила `[230, 150, 270, 120] = 770`. Це виправить відсікання (приклад 3.4);
- рядків 15 — майже всі `2⁴ = 16` підмножин; бракує лише `[230, 270, 120]`, бо на `[230, 270]` гілка зупинилась з відповіддю.

### Приклад 3.3. Без цілі: усі підмножини

Якщо прибрати умову `total == target`, перебір з поверненням видасть **усі** набори — кожна страва «є» чи «немає».

In [ ]:
def all_subsets(items):
    found = []

    def explore(start, chosen):
        found.append(list(chosen))             # кожен вузол дерева — окремий набір
        for i in range(start, len(items)):
            chosen.append(items[i])
            explore(i + 1, chosen)
            chosen.pop()

    explore(0, [])
    return found

In [ ]:
for subset in all_subsets(["Борщ", "Чай", "Хліб"]):
    print(subset)
for n in (3, 10, 20):
    print(f"елементів: {n:>2} → наборів: {len(all_subsets(list(range(n))))}")

3 страви — 8 наборів (від порожнього до всіх трьох), 10 — 1024, 20 — 1 048 576. Це `2ⁿ`: кожен новий елемент **подвоює** кількість варіантів. Тому повний перебір годиться лише для десятків елементів — і тому так важливо відсікати гілки.

### Приклад 3.4. Відсікання

Якщо відсортувати поїздки за зростанням, то, щойно чергова поїздка перевищує залишок, **усі наступні** перевищать теж. Ці гілки можна не досліджувати зовсім — це **відсікання** (pruning).

In [ ]:
def voucher_sets_pruned(fares, target):
    fares = sorted(fares)
    found = []
    calls = [0]

    def explore(start, chosen, total):
        calls[0] += 1
        if total == target:
            found.append(list(chosen))
            return
        for i in range(start, len(fares)):
            if total + fares[i] > target:
                break
            chosen.append(fares[i])
            explore(i + 1, chosen, total + fares[i])
            chosen.pop()

    explore(0, [], 0)
    return found, calls[0]


big_shift = [230, 150, 270, 180, 120, 410, 95, 60, 310, 200, 140, 175]

**🔮 Прогноз:** у скільки разів менше викликів дасть відсікання на 12 поїздках?

In [ ]:
print(voucher_sets_pruned(trip_fares, 500))
print(voucher_sets(big_shift, 500)[1], voucher_sets_pruned(big_shift, 500)[1])

<details>
<summary>Відповідь</summary>

На 6 поїздках — 19 викликів замість 56, на 12 — 150 замість 3454: більш ніж у 20 разів. Набори ті самі, лише записані за зростанням. Відсікання не змінює найгіршого випадку (`2ⁿ`), але на реальних даних прибирає більшість гілок.

</details>

### Приклад 3.5. `break` без сортування — втрачені відповіді

Відсікання `break` означає «наступні теж не влізуть». Що, якщо забути `sorted`?

In [ ]:
def voucher_sets_unsorted_break(fares, target):
    found = []

    def explore(start, chosen, total):
        if total == target:
            found.append(list(chosen))
            return
        for i in range(start, len(fares)):
            if total + fares[i] > target:
                break                      # «наступні теж не влізуть» — але список не відсортовано!
            chosen.append(fares[i])
            explore(i + 1, chosen, total + fares[i])
            chosen.pop()

    explore(0, [], 0)
    return found

In [ ]:
print("без сортування:", voucher_sets_unsorted_break(trip_fares, 500))
print("з сортуванням: ", voucher_sets_pruned(trip_fares, 500)[0])

Набір 230 + 150 + 120 зник: після `[230, 150]` (380) наступна поїздка 270 не влізла, `break` обірвав цикл — і до 120, яка стоїть **далі**, справа не дійшла. На невідсортованих даних можна лише `continue` (пропустити одну поїздку), а не `break`.

### Приклад 3.6. Ще одне відсікання: не більше k поїздок

Бухгалтерія дозволяє розділити ваучер щонайбільше на дві поїздки. Друге правило — ще одна умова зупинки гілки.

In [ ]:
def voucher_sets_limited(fares, target, max_trips):
    fares = sorted(fares)
    found = []
    calls = [0]

    def explore(start, chosen, total):
        calls[0] += 1
        if total == target:
            found.append(list(chosen))
            return
        if len(chosen) == max_trips:       # гілка вичерпала ліміт поїздок — далі не йдемо
            return
        for i in range(start, len(fares)):
            if total + fares[i] > target:
                break
            chosen.append(fares[i])
            explore(i + 1, chosen, total + fares[i])
            chosen.pop()

    explore(0, [], 0)
    return found, calls[0]

In [ ]:
for k in (2, 3, 12):
    found, calls = voucher_sets_limited(big_shift, 500, k)
    print(f"не більше {k:>2} поїздок: наборів — {len(found)}, викликів — {calls}")

Не більше двох поїздок дає лише один набір — `[230, 270]`, і вдвічі менше викликів. Чим жорсткіше обмеження, тим менше гілок і викликів. З лімітом 12 (усі поїздки) — ті самі 150 викликів, що й у прикладі 3.4. Кожне правило задачі, яке можна перевірити **до** кінця гілки, — шанс її відсікти.

### Задача 3. Обід рівно на 150 грн

Клієнт хоче обід рівно на `budget` гривень зі страв меню, кожна не більше одного разу. Допиши `lunch_sets(menu, budget)` — усі такі набори **назв** страв.

- список `items` пар `(назва, ціна)` у порядку меню вже зібрано генератором `dishes` з **задачі 1**;
- перебір з поверненням, як у прикладі 3.1: `chosen.append` → рекурсія `explore(i + 1, …)` → `chosen.pop()`;
- відсікання: страву, з якою сума перевищить бюджет, пропускай. Порядок — як у меню, **не** відсортований, тож `continue`, а не `break` (приклад 3.5);
- у `found` додавай список назв.

Для 150 грн наборів шість:

```text
['Борщ', 'Лате']
['Юшка', 'Чай', 'Узвар']
['Юшка', 'Лате', 'Хліб']
['Вареники', 'Чай', 'Еспресо']
['Деруни', 'Чай', 'Узвар', 'Хліб']
['Деруни', 'Еспресо', 'Узвар']
```

Перевірка — у наступній клітинці.

In [ ]:
def lunch_sets(menu, budget):
    items = [(path[-1], price) for path, price in dishes(menu)]
    found = []
    # YOUR CODE HERE
    pass
    return found

In [ ]:
result = lunch_sets(MENU, 150)
for names in result:
    print(names)
assert result == [["Борщ", "Лате"], ["Юшка", "Чай", "Узвар"], ["Юшка", "Лате", "Хліб"],
                  ["Вареники", "Чай", "Еспресо"], ["Деруни", "Чай", "Узвар", "Хліб"],
                  ["Деруни", "Еспресо", "Узвар"]]
assert len(lunch_sets(MENU, 200)) == 10 and lunch_sets(MENU, 5) == []
print("✅ Задача 3 розв'язана")

---
## 4. Коли рекурсія повторює роботу

Рекурсія природна, коли підзадачі **незалежні**: ліва й права половина злиття, різні категорії меню. Але якщо ті самі підзадачі трапляються знову і знову, рекурсія перераховує їх щоразу. Класичний приклад — числа Фібоначчі: `fib(n) = fib(n - 1) + fib(n - 2)`.

### Приклад 4.1. Скільки викликів робить `fib(20)`

In [ ]:
calls = [0]


def fib(n):
    calls[0] += 1
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

**🔮 Прогноз:** скільки викликів зробить `fib(20)` — порядку 20, 200 чи 20 000?

In [ ]:
print(fib(20), calls[0])

<details>
<summary>Відповідь</summary>

21 891 виклик, щоб отримати 6765. Звідки стільки — показують приклади 4.2 і 4.3.

</details>

### Приклад 4.2. Траса `fib(5)`: ті самі виклики знову й знову

In [ ]:
def fib_traced(n, depth=0):
    print("    " * depth + f"fib({n})")
    if n < 2:
        return n
    return fib_traced(n - 1, depth + 1) + fib_traced(n - 2, depth + 1)

In [ ]:
print("Результат:", fib_traced(5))

`fib(3)` у трасі з'являється двічі, `fib(2)` — тричі, `fib(1)` — п'ять разів. Кожна гілка рахує своє, не знаючи, що сусідня гілка вже порахувала те саме. На відміну від меню, тут підзадачі **перекриваються**.

### Приклад 4.3. Скільки разів рахується кожне `fib(k)`

In [ ]:
from collections import Counter

seen = Counter()


def fib_counted(n):
    seen[n] += 1
    if n < 2:
        return n
    return fib_counted(n - 1) + fib_counted(n - 2)

In [ ]:
fib_counted(20)
for k in (20, 19, 18, 15, 10, 5, 2, 1):
    print(f"fib({k:>2}): обчислень — {seen[k]:>5}")
print("Усього викликів:", sum(seen.values()))

`fib(20)` і `fib(19)` — по разу, `fib(18)` — двічі, `fib(15)` — 8 разів, а `fib(1)` — 6765. Кількість повторів сама росте як числа Фібоначчі — тобто експоненційно. Разом ті самі 21 891.

### Приклад 4.4. `@lru_cache` запам'ятовує відповіді

`@lru_cache` з уроку 10 зберігає результат для кожного аргументу. Повторний виклик з тим самим `n` бере відповідь з пам'яті.

In [ ]:
from functools import lru_cache


@lru_cache(maxsize=None)
def fib_memo(n):
    if n < 2:
        return n
    return fib_memo(n - 1) + fib_memo(n - 2)

In [ ]:
print(fib_memo(20), fib_memo.cache_info())
print(fib_memo(80))

`misses=21` — справжніх обчислень 21 (для `n` від 0 до 20), `hits=18` — стільки разів відповідь узяли з кешу. А `fib(80)` без кешу рахувався б роками; з кешем — миттєво. Це **мемоізація** — перший крок до динамічного програмування (урок 27).

### Приклад 4.5. Мемоізація вручну — словник

Що робить `lru_cache` всередині? По суті — словник «аргумент → відповідь», який перевіряють перед обчисленням.

In [ ]:
def fib_dict(n, memo=None):
    if memo is None:                   # новий словник на кожен зовнішній виклик (не memo={}!)
        memo = {}
    if n in memo:
        return memo[n]
    result = n if n < 2 else fib_dict(n - 1, memo) + fib_dict(n - 2, memo)
    memo[n] = result
    return result

In [ ]:
memo = {}
print(fib_dict(20, memo), "| відповідей у словнику:", len(memo))
print(fib_dict(30))

21 відповідь у словнику — стільки ж, скільки `misses` у `lru_cache`. `memo=None` замість `memo={}` — та сама пересторога, що в прикладі 1.6: значення за замовчуванням створюється один раз.

---
## 5. Рекурсія чи цикл: явний стек

Кожну рекурсію можна переписати циклом з **явним стеком** — списком, у який кладемо «що ще треба обробити». Стек викликів Python ми просто замінюємо своїм списком.

### Приклад 5.1. Підрахунок страв циклом

In [ ]:
def count_dishes_iterative(menu):
    stack = [menu]
    count = 0
    while stack:
        node = stack.pop()
        if isinstance(node, dict):
            stack.extend(node.values())
        else:
            count += 1
    return count

In [ ]:
print(count_dishes_iterative(MENU), count_dishes(MENU))

Та сама відповідь — 9. Замість «виклич себе для кожної частини» — «поклади частини в стек і бери звідти, доки він не порожній».

### Приклад 5.2. Пошук шляху циклом

Для шляху в стек кладемо пару «вузол + шлях до нього» — те, що в рекурсії жило в параметрі `path`.

In [ ]:
def find_path_iterative(menu, dish):
    stack = [(menu, ())]
    while stack:
        node, path = stack.pop()
        if not isinstance(node, dict):
            continue
        for name, child in node.items():
            if name == dish:
                return path + (name,)
            stack.append((child, path + (name,)))
    return None

In [ ]:
for dish in ("Узвар", "Лате", "Піца"):
    print(dish, "→", find_path_iterative(MENU, dish), "| рекурсія:", find_path(MENU, dish))

Відповіді збігаються з рекурсивною версією: у меню кожна страва трапляється один раз, тож порядок обходу на результат не впливає. Але код довший: стан, який рекурсія тримала в кадрах сама, тут доводиться зберігати вручну.

### Приклад 5.3. Дуже глибоке меню

Згенеруємо меню з 20 000 вкладених категорій (так виглядав би «JSON-ланцюжок» з помилкою експорту) і порахуємо страви обома способами.

In [ ]:
def deep_menu(levels):
    menu = {"Страва": 10}
    for level in range(levels):
        menu = {f"Рівень {level}": menu}
    return menu


DEEP = deep_menu(20_000)

In [ ]:
try:
    print("рекурсія:", count_dishes(DEEP))
except RecursionError:
    print("рекурсія: RecursionError — глибина більша за ліміт стеку")
print("цикл:", count_dishes_iterative(DEEP))

Рекурсія впала: 20 000 рівнів — більше за ліміт стеку (`sys.getrecursionlimit()`, приклад 1.7). Цикл з явним стеком упорався, бо його стек — звичайний список, обмежений лише пам'яттю.

| | Рекурсія | Цикл з явним стеком |
|---|---|---|
| Читабельність | повторює визначення задачі: «сума по частинах» | стек і цикл видно явно, ідея — менше |
| Глибина | обмежена лімітом стеку, 1000–3000 кадрів (`RecursionError`) | обмежена лише пам'яттю |
| Повернення з кроку | природне: стек викликів сам пам'ятає, де ми | треба зберігати стан у стеку вручну |
| Коли | дерева, вкладені структури, «розділяй і володарюй», перебір | лінійні дані, дуже глибокі структури, продакшн-обхід великих дерев |

Як обрати техніку:

```text
задача складається з менших копій себе?
├── ні  → цикл, словник (П1–П3)
└── так → підзадачі повторюються?
          ├── так → мемоізація / ДП (урок 27)
          └── ні  → треба перебрати варіанти вибору?
                    ├── так → перебір з поверненням + відсікання
                    └── ні  → розділяй і володарюй або обхід дерева
```

## ✅ Самоперевірка

1. Що буде без базового випадку? А якщо рекурсивний виклик не зменшує задачу?
2. Чому `find_path` передає шлях кортежем `path + (name,)`, а не списком з `append`?
3. Звідки `log n` у сортуванні злиттям?
4. Чому для максимуму «розділяй і володарюй» не дає виграшу, а для сортування — дає?
5. Що робить `chosen.pop()` у переборі з поверненням?
6. Чому відсікання `break` правильне лише для відсортованого списку?
7. Що змінює `lru_cache` для `fib`?
8. Меню з 5 рівнями і ланцюжок з 20 000 рівнів: де рекурсія доречна, а де — явний стек?

<details>
<summary>Відповіді</summary>

1. `RecursionError`: функція викликає себе, доки не закінчиться стек. Те саме, якщо задача не зменшується — базового випадку не досягнемо.
2. Кожен рівень створює новий кортеж, і гілки не діляться одним об'єктом. Спільний список накопичує імена всіх гілок і навіть попередніх викликів.
3. Поділ навпіл дає `log₂ n` рівнів; на кожному злиття проходить n елементів.
4. Максимум однаково мусить побачити кожен елемент (n − 1 порівнянь). Сортування виграє, бо злиття відсортованих половин коштує лише `O(n)`.
5. Прибирає останній вибір, щоб спробувати інший на його місці.
6. «Наступні теж не влізуть» — правда, лише якщо наступні не менші за поточний; інакше губляться відповіді (приклад 3.5).
7. Кожне `fib(k)` рахується один раз: 21 обчислення замість 21 891.
8. Меню — рекурсія природна. 20 000 рівнів — явний стек: рекурсія впаде з `RecursionError`.

</details>

### Шпаргалка

```python
def solve(node):                   # рекурсія
    if базовий_випадок(node):
        return відповідь
    return згорнути(solve(частина) for частина in частини(node))

def walk(node, path=()):           # шлях — кортеж, новий на кожному рівні
    ... walk(child, path + (name,))

def merge_sort(items):             # розділяй і володарюй
    if len(items) <= 1: return items
    mid = len(items) // 2
    return merge(merge_sort(items[:mid]), merge_sort(items[mid:]))

def explore(start, chosen, total): # перебір з поверненням
    if total == target: found.append(list(chosen)); return
    for i in range(start, len(items)):
        if total + items[i] > target: break      # відсікання (лише відсортовано!)
        chosen.append(items[i]); explore(i + 1, chosen, total + items[i]); chosen.pop()

@lru_cache(maxsize=None)           # підзадачі повторюються

stack = [root]                     # цикл замість рекурсії
while stack:
    node = stack.pop(); stack.extend(частини(node))
```

## Далі

- **Урок 24 — `@property`, декоратори класів, dunder**: `sorted(deliveries)` і `len(menu)` для наших класів.
- **Практикум П5 (урок 27)**: динамічне програмування.